# Homework 4 - Computing Point-in-Time Residual Returns
In this homework, we will use regressions to compute beta-adjusted "residual" returns in a point-in-time fashion suitable for backtesting / live trading.


1. Download Daily Bars for FB, AAPL, AMZN, NFLX, GOOGL and QQQ from yahoo finance starting 2016-01-01. Use the Adj Close to compute daily returns.
2. Now, let's compute the beta of FB, AAPL, AMZN, NFLX, GOOGL using QQQ as our benchmark. You can think of this as the beta these stocks have to their industry (tech). In practice,  we have to use some lookback window to compute the beta. Let's use 252 (1 year, excluding wknds/holidays). So, for each day, the betas should be computed using the most recent 252 data points.
3. Using the betas, compute an "alpha" on each day. This is also known as a "residual return".
4. Compare the volatility of the residual returns to that of the original returns. What do you notice?
5. Compare the pairwise correlations of the residual returns to that of the original returns. What do you notice?
6. Compute the information ratio for each of these stocks and compare that to the sharpe ratio.


In [14]:
import yfinance as yf
import numpy as np
import pandas as pd

tickerlist = ['META','AAPL','AMZN','NFLX','GOOGL','QQQ']
tickerdata = yf.download(tickerlist, start="2016-01-01")

[*********************100%***********************]  6 of 6 completed


In [15]:
print(type(tickerdata))
print(tickerdata.shape)
print(tickerdata.columns)

<class 'pandas.core.frame.DataFrame'>
(2525, 30)
MultiIndex([( 'Close',  'AAPL'),
            ( 'Close',  'AMZN'),
            ( 'Close', 'GOOGL'),
            ( 'Close',  'META'),
            ( 'Close',  'NFLX'),
            ( 'Close',   'QQQ'),
            (  'High',  'AAPL'),
            (  'High',  'AMZN'),
            (  'High', 'GOOGL'),
            (  'High',  'META'),
            (  'High',  'NFLX'),
            (  'High',   'QQQ'),
            (   'Low',  'AAPL'),
            (   'Low',  'AMZN'),
            (   'Low', 'GOOGL'),
            (   'Low',  'META'),
            (   'Low',  'NFLX'),
            (   'Low',   'QQQ'),
            (  'Open',  'AAPL'),
            (  'Open',  'AMZN'),
            (  'Open', 'GOOGL'),
            (  'Open',  'META'),
            (  'Open',  'NFLX'),
            (  'Open',   'QQQ'),
            ('Volume',  'AAPL'),
            ('Volume',  'AMZN'),
            ('Volume', 'GOOGL'),
            ('Volume',  'META'),
            ('Volume',  'NF

In [19]:
tickerdata.to_pickle("tech_raw_2016.pkl")
print(tickerdata)

Price            Close                                                 \
Ticker            AAPL        AMZN       GOOGL        META       NFLX   
Date                                                                    
2016-01-04   23.753155   31.849501   37.687248  101.510918  10.996000   
2016-01-05   23.157913   31.689501   37.790962  102.017387  10.766000   
2016-01-06   22.704721   31.632500   37.681786  102.255722  11.768000   
2016-01-07   21.746473   30.396999   36.772160   97.240730  11.456000   
2016-01-08   21.861471   30.352501   36.271446   96.654846  11.139000   
...                ...         ...         ...         ...        ...   
2026-01-12  260.250000  246.470001  331.859985  641.969971  89.410004   
2026-01-13  261.049988  242.600006  335.970001  631.090027  90.320000   
2026-01-14  259.959991  236.649994  335.839996  615.520020  88.550003   
2026-01-15  258.209991  238.179993  332.779999  620.799988  88.050003   
2026-01-16  255.529999  239.119995  330.000000  620

In [21]:
calculated_returns = tickerdata['Close'] / tickerdata['Close'].shift() - 1
print(calculated_returns)

Ticker          AAPL      AMZN     GOOGL      META      NFLX       QQQ
Date                                                                  
2016-01-04       NaN       NaN       NaN       NaN       NaN       NaN
2016-01-05 -0.025059 -0.005024  0.002752  0.004989 -0.020917 -0.001735
2016-01-06 -0.019570 -0.001799 -0.002889  0.002336  0.093071 -0.009606
2016-01-07 -0.042205 -0.039058 -0.024140 -0.049044 -0.026513 -0.031313
2016-01-08  0.005288 -0.001464 -0.013617 -0.006025 -0.027671 -0.008201
...              ...       ...       ...       ...       ...       ...
2026-01-12  0.003393 -0.003679  0.010013 -0.016982 -0.000559  0.000830
2026-01-13  0.003074 -0.015702  0.012385 -0.016948  0.010178 -0.001483
2026-01-14 -0.004175 -0.024526 -0.000387 -0.024672 -0.019597 -0.010683
2026-01-15 -0.006732  0.006465 -0.009111  0.008578 -0.005647  0.003599
2026-01-16 -0.010379  0.003947 -0.008354 -0.000886 -0.000568 -0.000836

[2525 rows x 6 columns]


In [25]:
days = 252
correlation = calculated_returns.rolling(days).corr(calculated_returns['QQQ'])
print(correlation)

Ticker          AAPL      AMZN     GOOGL      META      NFLX  QQQ
Date                                                             
2016-01-04       NaN       NaN       NaN       NaN       NaN  NaN
2016-01-05       NaN       NaN       NaN       NaN       NaN  NaN
2016-01-06       NaN       NaN       NaN       NaN       NaN  NaN
2016-01-07       NaN       NaN       NaN       NaN       NaN  NaN
2016-01-08       NaN       NaN       NaN       NaN       NaN  NaN
...              ...       ...       ...       ...       ...  ...
2026-01-12  0.721989  0.766532  0.661921  0.730643  0.509352  1.0
2026-01-13  0.720338  0.765688  0.660688  0.733834  0.505141  1.0
2026-01-14  0.720029  0.766339  0.660147  0.734546  0.506822  1.0
2026-01-15  0.719522  0.766354  0.659337  0.735528  0.506641  1.0
2026-01-16  0.718208  0.764788  0.656305  0.732980  0.503512  1.0

[2525 rows x 6 columns]


In [26]:
volatility = calculated_returns.rolling(days).std()
print(volatility)

Ticker          AAPL      AMZN     GOOGL      META      NFLX       QQQ
Date                                                                  
2016-01-04       NaN       NaN       NaN       NaN       NaN       NaN
2016-01-05       NaN       NaN       NaN       NaN       NaN       NaN
2016-01-06       NaN       NaN       NaN       NaN       NaN       NaN
2016-01-07       NaN       NaN       NaN       NaN       NaN       NaN
2016-01-08       NaN       NaN       NaN       NaN       NaN       NaN
...              ...       ...       ...       ...       ...       ...
2026-01-12  0.020377  0.021738  0.020285  0.023711  0.021397  0.014743
2026-01-13  0.020318  0.021742  0.020280  0.023731  0.021233  0.014706
2026-01-14  0.020308  0.021799  0.020275  0.023770  0.021270  0.014722
2026-01-15  0.020311  0.021801  0.020279  0.023729  0.021253  0.014723
2026-01-16  0.020287  0.021745  0.020210  0.023607  0.021201  0.014657

[2525 rows x 6 columns]


In [27]:
beta = (correlation*volatility).divide(volatility['QQQ'],axis=0)  
print(beta)

Ticker          AAPL      AMZN     GOOGL      META      NFLX  QQQ
Date                                                             
2016-01-04       NaN       NaN       NaN       NaN       NaN  NaN
2016-01-05       NaN       NaN       NaN       NaN       NaN  NaN
2016-01-06       NaN       NaN       NaN       NaN       NaN  NaN
2016-01-07       NaN       NaN       NaN       NaN       NaN  NaN
2016-01-08       NaN       NaN       NaN       NaN       NaN  NaN
...              ...       ...       ...       ...       ...  ...
2026-01-12  0.997898  1.130272  0.910778  1.175111  0.739259  1.0
2026-01-13  0.995207  1.132034  0.911119  1.184179  0.729324  1.0
2026-01-14  0.993235  1.134738  0.909152  1.186000  0.732235  1.0
2026-01-15  0.992624  1.134822  0.908198  1.185505  0.731383  1.0
2026-01-16  0.994080  1.134631  0.904965  1.180602  0.728345  1.0

[2525 rows x 6 columns]


In [30]:
residual_alpha = calculated_returns - beta.multiply(calculated_returns['QQQ'],0)
print(residual_alpha)

Ticker          AAPL      AMZN     GOOGL      META      NFLX           QQQ
Date                                                                      
2016-01-04       NaN       NaN       NaN       NaN       NaN           NaN
2016-01-05       NaN       NaN       NaN       NaN       NaN           NaN
2016-01-06       NaN       NaN       NaN       NaN       NaN           NaN
2016-01-07       NaN       NaN       NaN       NaN       NaN           NaN
2016-01-08       NaN       NaN       NaN       NaN       NaN           NaN
...              ...       ...       ...       ...       ...           ...
2026-01-12  0.002565 -0.004616  0.009257 -0.017957 -0.001172  3.252607e-19
2026-01-13  0.004550 -0.014023  0.013736 -0.015192  0.011259 -4.336809e-19
2026-01-14  0.006435 -0.012404  0.009325 -0.012002 -0.011775 -3.469447e-18
2026-01-15 -0.010305  0.002381 -0.012380  0.004311 -0.008279  2.168404e-18
2026-01-16 -0.009548  0.004896 -0.007597  0.000101  0.000041 -3.252607e-19

[2525 rows x 6 columns]


In [31]:
volatility_structure = {}
volatility_structure['original'] = calculated_returns.std()*np.sqrt(days)
volatility_structure['residual'] = residual_alpha.std()*np.sqrt(days)
volatility_structure = pd.DataFrame(volatility_structure).drop('QQQ')
volatility_structure

,original,residual
Ticker,,
AAPL,0.290040,0.169871
AMZN,0.327827,0.204055
GOOGL,0.287917,0.182281
META,0.384611,0.276021
NFLX,0.418837,0.327680


In [32]:
calculated_returns.corr()

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Ticker,,,,,,
AAPL,1.000000,0.572546,0.609572,0.521425,0.431052,0.806206
AMZN,0.572546,1.000000,0.635576,0.608341,0.523871,0.767595
GOOGL,0.609572,0.635576,1.000000,0.611871,0.441451,0.785335
META,0.521425,0.608341,0.611871,1.000000,0.459538,0.701224
NFLX,0.431052,0.523871,0.441451,0.459538,1.000000,0.586046
QQQ,0.806206,0.767595,0.785335,0.701224,0.586046,1.000000


In [33]:
residual_alpha.corr()

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Ticker,,,,,,
AAPL,1.000000,-0.100145,-0.060806,-0.087576,-0.092762,-0.013001
AMZN,-0.100145,1.000000,0.060514,0.124912,0.131948,-0.028511
GOOGL,-0.060806,0.060514,1.000000,0.123256,-0.063531,-0.040288
META,-0.087576,0.124912,0.123256,1.000000,0.083994,-0.031142
NFLX,-0.092762,0.131948,-0.063531,0.083994,1.000000,-0.028026
QQQ,-0.013001,-0.028511,-0.040288,-0.031142,-0.028026,1.000000


In [34]:
final_data_frame = {}
final_data_frame['IR'] = residual_alpha.mean() / residual_alpha.std()*np.sqrt(days)
final_data_frame['SR'] = calculated_returns.mean() / calculated_returns.std()*np.sqrt(days)
final_data_frame = pd.DataFrame(final_data_frame).drop('QQQ')
final_data_frame

,IR,SR
Ticker,,
AAPL,0.336405,0.962913
AMZN,0.053436,0.777716
GOOGL,0.306267,0.896699
META,0.025966,0.664186
NFLX,0.206208,0.709679
